# RAG Chain
**Purpose:** Build the full Retrieval-Augmented Generation pipeline.
Given a user query, the system retrieves the most relevant events 
from the FAISS index and generates a contextual response using 
Mistral LLM.

**Components:**
- Retriever: FAISS semantic search
- Prompt: system prompt constraining LLM to context only
- Generator: Mistral LLM (mistral-small-latest)
- Orchestration: LangChain RetrievalQA

**Input:**
- `data/processed/faiss_index.idx`
- `data/processed/metadata.json`

**Author:** Hope  
**Date:** 2026-03-23

## 1.1 Imports and environment validation

In [1]:
import json
import numpy as np
import faiss
from pathlib import Path
from dotenv import load_dotenv
import os
from mistralai import Mistral

load_dotenv(dotenv_path=Path("../.env"))

mistral_key = os.getenv("MISTRAL_API_KEY")
assert mistral_key is not None, "MISTRAL_API_KEY not found in .env"

client = Mistral(api_key=mistral_key)

print("Environment loaded successfully.")
print(f"Mistral client initialised: {type(client)}")

Environment loaded successfully.
Mistral client initialised: <class 'mistralai.sdk.Mistral'>


## 1.2 Configuration

In [2]:
PROCESSED_DIR = Path("../data/processed")
INDEX_PATH    = PROCESSED_DIR / "faiss_index.idx"
META_PATH     = PROCESSED_DIR / "metadata.json"

EMBEDDING_MODEL = "mistral-embed"
LLM_MODEL       = "mistral-small-latest"
LLM_TEMPERATURE = 0.1
TOP_K           = 5

print("Configuration set:")
print(f"  Index path      : {INDEX_PATH}")
print(f"  Metadata path   : {META_PATH}")
print(f"  Embedding model : {EMBEDDING_MODEL}")
print(f"  LLM model       : {LLM_MODEL}")
print(f"  Temperature     : {LLM_TEMPERATURE}")
print(f"  Top-k results   : {TOP_K}")

Configuration set:
  Index path      : ../data/processed/faiss_index.idx
  Metadata path   : ../data/processed/metadata.json
  Embedding model : mistral-embed
  LLM model       : mistral-small-latest
  Temperature     : 0.1
  Top-k results   : 5


## 1.3 Load FAISS index and metadata

In [3]:
index = faiss.read_index(str(INDEX_PATH))

with open(META_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)

print(f"Index loaded      : {index.ntotal} vectors")
print(f"Metadata loaded   : {len(metadata)} records")

assert index.ntotal == len(metadata), \
    f"Mismatch: {index.ntotal} vectors vs {len(metadata)} metadata records"

print("\nAll assertions passed.")

Index loaded      : 1740 vectors
Metadata loaded   : 1740 records

All assertions passed.


## 1.4 Core RAG functions

In [4]:
def embed_query(text: str) -> np.ndarray:
    """Embed a single user query."""
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        inputs=[text]
    )
    return np.array([response.data[0].embedding], dtype="float32")


# --- 2. Retriever ---
def retrieve(query: str, k: int = TOP_K) -> list[dict]:
    """Search FAISS index and return top-k matching event metadata."""
    query_vec            = embed_query(query)
    distances, indices   = index.search(query_vec, k)

    results = []
    for idx, dist in zip(indices[0], distances[0]):
        result = metadata[idx].copy()
        result["distance"] = float(dist)
        results.append(result)
    return results


# --- 3. System prompt ---
SYSTEM_PROMPT = """Tu es un assistant spécialisé dans les événements culturels à Paris.
Tu aides les utilisateurs à découvrir des événements en te basant UNIQUEMENT sur 
les informations fournies dans le contexte ci-dessous.

Règles strictes :
- Réponds UNIQUEMENT en te basant sur les événements fournis dans le contexte.
- Si l'information n'est pas dans le contexte, dis clairement que tu ne sais pas.
- Ne génère JAMAIS d'informations inventées sur des événements.
- Réponds toujours en français.
- Sois concis, précis et utile.
- Pour chaque événement mentionné, indique le titre, la date et le lieu.
"""


# --- 4. Generator ---
def generate(query: str, context_events: list[dict]) -> str:
    """Generate a response using Mistral LLM with retrieved context."""

    # Format context from retrieved events
    context = "\n\n".join([
        f"Événement {i+1}:\n{event['text']}"
        for i, event in enumerate(context_events)
    ])

    user_message = f"""Contexte des événements disponibles :
{context}

Question de l'utilisateur : {query}

Réponds en te basant uniquement sur les événements fournis ci-dessus."""

    response = client.chat.complete(
        model=LLM_MODEL,
        temperature=LLM_TEMPERATURE,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_message}
        ]
    )
    return response.choices[0].message.content


# --- 5. Full RAG pipeline ---
def rag(query: str, k: int = TOP_K) -> dict:
    """Full RAG pipeline: retrieve + generate."""
    retrieved = retrieve(query, k)
    answer    = generate(query, retrieved)
    return {
        "query"    : query,
        "answer"   : answer,
        "sources"  : retrieved
    }

print("RAG functions defined successfully.")
print("Functions available: embed_query, retrieve, generate, rag")

RAG functions defined successfully.
Functions available: embed_query, retrieve, generate, rag


## 1.5  Layer by layer testing

In [5]:
print("=== TEST 1: Retriever ===")
test_query  = "Concert de musique classique à Paris"
results     = retrieve(test_query)

print(f"Query   : {test_query}")
print(f"Results : {len(results)}")
for i, r in enumerate(results, 1):
    print(f"\n  {i}. {r['title']}")
    print(f"     Date    : {r['date_range']}")
    print(f"     City    : {r['city']}")
    print(f"     Distance: {r['distance']:.4f}")

=== TEST 1: Retriever ===
Query   : Concert de musique classique à Paris
Results : 5

  1. Concert de Printemps France Asie
     Date    : Dimanche 8 mars, 15h30
     City    : Paris
     Distance: 0.4266

  2. Concert
     Date    : Dimanche 29 juin 2025, 16h00
     City    : Paris
     Distance: 0.4280

  3. Les 4 Saisons de Vivaldi, Petite Musique de Nuit de Mozart​
     Date    : Mercredi 11 mars, 20h00
     City    : Paris
     Distance: 0.4296

  4. Les 4 Saisons de Vivaldi, Petite Musique de Nuit de Mozart
     Date    : Mardi 7 avril, 20h00
     City    : Paris
     Distance: 0.4304

  5. Les 4 Saisons de Vivaldi et la Petite Musique de Nuit de Mozart
     Date    : Samedi 5 juillet 2025, 20h45
     City    : Paris
     Distance: 0.4322


## 1.5 Full RAG pipeline 

In [6]:
print("=== TEST 2: Full RAG Pipeline ===")

result = rag("Quels concerts de musique classique ont lieu à Paris ?")

print(f"Query  : {result['query']}")
print(f"\nAnswer :\n{result['answer']}")
print(f"\nSources used: {len(result['sources'])}")
for i, s in enumerate(result['sources'], 1):
    print(f"  {i}. {s['title']} — {s['date_range']}")

=== TEST 2: Full RAG Pipeline ===
Query  : Quels concerts de musique classique ont lieu à Paris ?

Answer :
Voici les concerts de musique classique à Paris disponibles dans le contexte :

1. **Les 4 Saisons de Vivaldi, Petite Musique de Nuit de Mozart**
   - **Date** : Mercredi 11 mars, 20h00
   - **Lieu** : Église de La Madeleine, Paris

2. **Concert spirituel**
   - **Date** : 30 novembre - 21 décembre 2025, les dimanches (après la messe de 11h)
   - **Lieu** : Basilique Sainte-Clotilde, Paris

3. **Les 4 Saisons de Vivaldi, Petite Musique de Nuit de Mozart**
   - **Date** : Samedi 28 mars, 20h00
   - **Lieu** : Église de La Madeleine, Paris

4. **Les 4 Saisons de Vivaldi, Petite Musique de Nuit de Mozart**
   - **Date** : Mardi 7 avril, 20h00
   - **Lieu** : Église de La Madeleine, Paris

5. **Concert Aria Baroque** (Vivaldi, Haendel et Bach)
   - **Date** : Samedi 14 mars, 16h00
   - **Lieu** : Église Sainte-Elisabeth de Hongrie, Paris

Sources used: 5
  1. Les 4 Saisons de Vivaldi

## 1.6 Now test two edge cases:

In [7]:
# --- Test 3: Out of scope query ---
print("=== TEST 3: Out of scope query ===")
result = rag("Quel est le meilleur restaurant à Paris ?")
print(f"Answer :\n{result['answer']}")

print("\n" + "="*50)

# --- Test 4: Vague query ---
print("\n=== TEST 4: Vague query ===")
result = rag("Que faire ce weekend à Paris ?")
print(f"Answer :\n{result['answer']}")

print("\n" + "="*50)

# --- Test 5: No match query ---
print("\n=== TEST 5: No match query ===")
result = rag("Y a-t-il des événements de corrida à Paris ?")
print(f"Answer :\n{result['answer']}")

=== TEST 3: Out of scope query ===
Answer :
Je ne sais pas. Les événements fournis ne mentionnent pas de restaurant à Paris.


=== TEST 4: Vague query ===
Answer :
Voici les événements disponibles ce weekend à Paris :

- **Soirée de partage pour les jeunes pros et étudiants**
  *Date* : Samedi 7 février, 19h00
  *Lieu* : Salle de La Madeleine, Paris

- **Ciné-Club** (film : *Invictus*)
  *Date* : Samedi 14 février, 19h00
  *Lieu* : Salle de La Madeleine, Paris


=== TEST 5: No match query ===
Answer :
Je ne sais pas.
